# Pamplona Gastronomic ChatBot

Final group delivery for AI Technology course - 2026

Andrés González

Xabier Ilarraz

Santiago Quintanilla

## 1. Install and import libraries, and enter the API key
Run this cell once. If asked to restart the notebook kernel, restart and continue from cell 2.

In [13]:
%pip -q install "langchain-community>=0.3,<1" "langchain-text-splitters>=0.3,<1" "langchain-huggingface>=0.1,<2" "langchain-chroma>=0.2,<2" "langchain-google-genai>=2,<4" "sentence-transformers>=3,<6" "pypdf>=5,<7" "ipywidgets>=8,<9"


In [14]:
import getpass
from pathlib import Path
from IPython.display import display, Markdown
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

api_key = getpass.getpass("Enter your Google Gemini API key: ").strip()
if not api_key:
    print("No API key entered. You can still run the retrieval experiments.")
else:
    print("API key received. It will not be displayed or saved in this notebook.")


Enter your Google Gemini API key: ··········
API key received. It will not be displayed or saved in this notebook.


## 2. Restaurant PDFs
You will need uploaded the 20 PDF files inside a `restaurant_docs` zip folder in this notebook.



In [15]:
import zipfile
from pathlib import Path

# El zip se extrae como 'Restaurant docs'
pdf_folder = Path("Restaurant docs")

# Si no existe, probamos con la alternativa en minúsculas
if not pdf_folder.exists():
    pdf_folder = Path("restaurant_docs")

zip_file = Path("restaurant_docs.zip")

# Si la carpeta no existe o está vacía, y el archivo zip existe, lo descomprimimos
if zip_file.exists() and (not pdf_folder.exists() or not list(pdf_folder.glob("*.pdf"))):
    print("Descomprimiendo restaurant_docs.zip...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(".")
    # Actualizamos la ruta por si se descomprimió con el nombre del zip o el interno
    if Path("Restaurant docs").exists():
        pdf_folder = Path("Restaurant docs")
    else:
        pdf_folder = Path("restaurant_docs")

pdf_files = list(pdf_folder.glob("*.pdf"))
print("Restaurant PDFs found:", len(pdf_files))
if not pdf_files:
    raise FileNotFoundError("Put the restaurant PDFs in the restaurant_docs folder.")

Restaurant PDFs found: 20


## 3. Load PDFs and split into chunks
We use chunks of 750 characters with an overlap of 120. The overlap helps avoid losing information when a sentence is split between chunks.

In [16]:
documents = []
for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    pages = PyPDFLoader(str(pdf_file)).load()
    for page in pages:
        page.metadata["restaurant"] = pdf_file.name
    documents.extend(pages)

chunk_size = 750
overlap = 120
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
chunks = splitter.split_documents(documents)
print("PDF pages:", len(documents))
print("Chunks:", len(chunks))

PDF pages: 24
Chunks: 96


## 4. Embeddings and vector database
We use a multilingual model because the documents and questions may be in Spanish or English. Chroma stores the vectors and helps find the most relevant chunks.

In [17]:
embedding_model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)
vector_db = Chroma.from_documents(chunks, embedding=embeddings, collection_name="pamplona_guide")
print("Vector database ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector database ready!


## 5. Chatbot: retrieval, prompt and simple memory
The model must use only the PDF extracts. Conversation history is used to understand follow-up questions, not as evidence about restaurants.

In [18]:
from IPython.display import display, Markdown

system_prompt = """You are a helpful restaurant guide for Pamplona.
Only use the PDF extracts below to answer questions about restaurants.
Do not invent facts or use outside information.
If the documents do not contain the answer, say: "I cannot confirm that from the provided restaurant documents."
If the question is unrelated to the restaurants, politely say that you only answer questions about these restaurants.
Do not claim real-time opening hours, prices or reservation availability.
When possible, mention the source document in your answer.
Use the same language as the user's question.
The chat history can help you understand a follow-up, but it is not a source of restaurant facts.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Recent conversation:\n{history}\n\nPDF extracts:\n{context}\n\nQuestion: {question}")
])

chat_history = []

def ask(question, show_sources=True):
    if not question.strip():
        print("Please write a question.")
        return

    # Use the previous question for follow-up questions
    previous_question = next(
        (m["content"] for m in reversed(chat_history) if m["role"] == "user"),
        ""
    )

    search_question = previous_question + " " + question if previous_question else question

    # Find relevant information in the PDFs
    found_docs = vector_db.similarity_search(search_question, k=4)

    if not found_docs:
        answer = "I cannot confirm that from the provided restaurant documents."

    elif not api_key:
        answer = "Please enter your Gemini API key in section 2."

    else:
        context = "\n\n".join(
            f"[{i}] {d.metadata.get('restaurant', 'unknown')}, page {d.metadata.get('page', 0)+1}: {d.page_content}"
            for i, d in enumerate(found_docs, 1)
        )

        history_text = "\n".join(
            f"{m['role']}: {m['content']}" for m in chat_history[-6:]
        )

        # Generate the answer with Gemini
        llm = ChatGoogleGenerativeAI(
            model="gemini-3.5-flash",
            google_api_key=api_key,
            temperature=0
        )

        try:
            result = (prompt | llm).invoke({
                "history": history_text,
                "context": context,
                "question": question
            })

            answer = result.content

            # Convert Gemini's response into normal text
            if isinstance(answer, list):
                answer = "\n".join(
                    item.get("text", "")
                    for item in answer
                    if isinstance(item, dict) and item.get("type") == "text"
                )

        except Exception as error:
            print(f"Error calling Gemini: {error}")
            return

    # Save conversation history
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

    # Display the answer with a clean format
    display(Markdown("### Answer\n\n" + answer))

    # Show the PDF sources
    if show_sources:
        print("\nRETRIEVED PDF SOURCES:")
        for d in found_docs:
            print(
                "-",
                d.metadata.get("restaurant", "unknown"),
                "(page", d.metadata.get("page", 0) + 1, ")"
            )


def clear_memory():
    chat_history.clear()
    print("Conversation memory cleared.")


print("Chatbot ready. Use ask('your question') in the next cell.")

Chatbot ready. Use ask('your question') in the next cell.


## 6. Try the chatbot
Change the question and run this cell again, or generate a new cell with the `ask("")` format. To test memory, ask a follow-up in another cell without calling `clear_memory()`.

In [19]:
ask("Which restaurants would you recommend for a special dinner in Pamplona?")

### Answer

Based on the provided documents, here are two great options in Pamplona for a special dinner:

1. **Restaurante Alhambra** (*19. Restaurante_Alhambra.pdf*): 
   It offers an atmosphere of sober elegance, comfort, and exclusivity, making it ideal for special celebrations and long business meals. Its high cuisine focuses on seasonal Navarrese products, adapting classic recipes with contemporary touches alongside attentive "white-glove" service.

2. **Restaurante Ábaco** (*8. Restaurante_Abaco.pdf*): 
   Located in the center of Pamplona, Ábaco provides a minimalist, sophisticated, and avant-garde atmosphere. It is a benchmark for high cuisine in miniature (*pinchos*), featuring a technical, conceptual, and aesthetic culinary style designed to offer sensory surprises.


RETRIEVED PDF SOURCES:
- 19. Restaurante_Alhambra.pdf (page 1 )
- 19. Restaurante_Alhambra.pdf (page 1 )
- 8. Restaurante_Abaco.pdf (page 1 )
- 8. Restaurante_Abaco.pdf (page 1 )


### Follow-up question (memory test)

In [ ]:
ask("¿Hay menú para celíacos en alguno?")

### Answer

No puedo confirmar eso a partir de los documentos de restaurantes proporcionados, ya que en los extractos disponibles no se menciona información sobre menús para celíacos u opciones sin gluten para ninguno de estos restaurantes.


RETRIEVED PDF SOURCES:
- 19. Restaurante_Alhambra.pdf (page 1 )
- 8. Restaurante_Abaco.pdf (page 1 )
- 10. Restaurante_Cocina_Vasca.pdf (page 1 )
- 17. Restaurante_Rodero.pdf (page 1 )


In [ ]:
ask("And what kind of food do they serve?")

### Answer

If you are asking about the restaurants mentioned in our previous conversation (Alhambra, Rodero, or Ábaco), **I cannot confirm that from the provided restaurant documents**, as their specific menus are not included in the available extracts.

However, based on the provided documents, here is the food served at the following restaurants:

* **La Loli Gastrobar**: Serves grilled meats (such as *Entraña a la Brasa* and *Matambre a la Pizza*), *Milanesa Completa*, homemade croquettes, and premium burgers (*6. La_Loli_Gastrobar.pdf*).
* **La Mudita**: Specializes in gourmet burgers (such as *La Txuleta Burger*, *Sweet & Sour Burger*, *Hamburguesa Gamberra*, and the vegetarian *Veggie Mudita*) as well as tempura-battered onion rings (*2. La Mudita.pdf*).
* **Sidrerías Toki Alai**: Offers traditional Basque cider house dishes, including *Tortilla de Txangurro* (spider crab omelette), *Tacos de Bacalao a la Vizcaína* (cod in Vizcaína sauce), green salad, Idiazábal cheese with quince and walnuts, and Tolosa tiles and cigarettes (*15. Sidrerias_Toki_Alai.pdf*).


RETRIEVED PDF SOURCES:
- 6. La_Loli_Gastrobar.pdf (page 1 )
- 2. La Mudita.pdf (page 1 )
- 15. Sidrerias_Toki_Alai.pdf (page 1 )
- 15. Sidrerias_Toki_Alai.pdf (page 1 )


### More test questions
These cover normal, unsupported, unrelated and nonsensical questions in both supported languages (spanish and english). We chose to run all of them individually to avoid unnecessary API costs.

In [ ]:
test_questions = [
    "Which restaurant offers Asian barbecue?",
    "Háblame del Restaurante Rodero.",
    "Which restaurant is the cheapest today?",
    "Can you book a table for tonight?",
    "Who won the football World Cup in 2010?",
    "Can I eat the moon at one of these restaurants?"
]

# Test all 6 questions
clear_memory()

for i, question in enumerate(test_questions, 1):
    print(f"\nQUESTION {i}: {question}")
    ask(question, show_sources=True)
    print("-" * 50)

Conversation memory cleared.

QUESTION 1: Which restaurant offers Asian barbecue?


### Answer

Based on the document **7. Hanazono_BBQ.pdf**, **Hanazono BBQ** offers Asian barbecue (specifically Korean-Japanese barbecue or Yakiniku). The restaurant features tables equipped with integrated grills and smoke extraction systems so diners can participate in cooking their food.


RETRIEVED PDF SOURCES:
- 7. Hanazono_BBQ.pdf (page 1 )
- 7. Hanazono_BBQ.pdf (page 1 )
- 10. Restaurante_Cocina_Vasca.pdf (page 1 )
- 2. La Mudita.pdf (page 1 )
--------------------------------------------------

QUESTION 2: Háblame del Restaurante Rodero.


### Answer

Basándome en el documento **17. Restaurante_Rodero.pdf**, el **Restaurante Rodero** está ubicado en el corazón de Pamplona y es uno de los referentes de la gastronomía de vanguardia en Navarra. 

Aquí tienes los principales detalles sobre el establecimiento:

* **Historia:** Fundado originalmente como una casa de comidas tradicional por la familia Rodero, el restaurante evolucionó hacia la alta cocina bajo la dirección técnica del chef Koldo Rodero y sus hermanas, Veronique y Goretti.
* **Ambiente:** Ofrece un espacio elegante, minimalista y sofisticado, diseñado específicamente para centrar la atención en la experiencia sensorial y el despliegue artístico de sus creaciones.
* **Propuesta culinaria:** Su cocina se caracteriza por una profunda innovación gastronómica estrechamente arraigada en la riqueza del producto.


RETRIEVED PDF SOURCES:
- 7. Hanazono_BBQ.pdf (page 1 )
- 7. Hanazono_BBQ.pdf (page 1 )
- 16. Cafe_Iruna.pdf (page 1 )
- 17. Restaurante_Rodero.pdf (page 1 )
--------------------------------------------------

QUESTION 3: Which restaurant is the cheapest today?


### Answer

I cannot confirm that from the provided restaurant documents.

... (3 outputs left out)

### Answer

I am sorry, but I only answer questions about these Pamplona restaurants.


RETRIEVED PDF SOURCES:
- 20. La_Cocina_de_Alex_Mugica.pdf (page 1 )
- 1. Babylon.pdf (page 1 )
- 3. More Than Burgers.pdf (page 1 )
- 13. Asador_Bidea_2.pdf (page 1 )
--------------------------------------------------

QUESTION 6: Can I eat the moon at one of these restaurants?


### Answer

I cannot confirm that from the provided restaurant documents.


RETRIEVED PDF SOURCES:
- 16. Cafe_Iruna.pdf (page 1 )
- 18. Restaurante_Europa.pdf (page 1 )
- 1. Babylon.pdf (page 1 )
- 19. Restaurante_Alhambra.pdf (page 1 )
--------------------------------------------------


## 7. Experiment: compare chunk sizes and top-k

We compare three chunk sizes and three retrieval settings. We check whether the correct restaurant PDF appears in the search results. This measures document retrieval, not the quality of the ChatBots's final answers.


In [ ]:
import csv

cases = [
    ("Which restaurant offers Asian barbecue with grills at the table?", "Hanazono_BBQ.pdf"),
    ("Which restaurant is led by chef Koldo Rodero?", "Restaurante_Rodero.pdf"),
    ("Where can I find Asador Bidea 2?", "Asador_Bidea_2.pdf"),
    ("Which restaurant is associated with chef Alex Mugica?", "La_Cocina_de_Alex_Mugica.pdf"),
]

results = []
for size, chunk_overlap in [(500, 80), (750, 120), (1000, 150)]:
    test_chunks = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=chunk_overlap
    ).split_documents(documents)
    test_db = Chroma.from_documents(test_chunks, embedding=embeddings, collection_name=f"test_{size}")
    for k in [2, 4, 6]:
        hits = 0
        for question, expected_file in cases:
            retrieved = test_db.similarity_search(question, k=k)
            if any(expected_file.lower() in d.metadata.get("restaurant", "").lower() for d in retrieved):
                hits += 1
        results.append({"chunk_size": size, "overlap": chunk_overlap,
                        "top_k": k, "correct": hits, "total": len(cases),
                        "hit_rate": round(hits / len(cases), 2)})
    test_db.delete_collection()

for row in results:
    print(row)

with open("experiment_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=results[0].keys())
    writer.writeheader()
    writer.writerows(results)
print("Results saved to experiment_results.csv")

## 8. ChatBot's characteristics

- **Chunking:** We tried 500, 750 and 1,000 characters. We used overlap so important information is not cut off. Explain what the experiment actually showed.
- **Embeddings:** We chose a multilingual model because users can ask in Spanish or English.
- **Database:** Chroma helps us search the PDF chunks by meaning, not just by exact words.
- **Memory:** We keep the last six chat messages and include the previous user question in the search. This is simple session memory, not permanent storage.
- **Reliability:** The prompt tells the chatbot not to invent restaurant facts. It can still make mistakes, so we test unsupported questions.
- **Evaluation:** We compare retrieval hit rates and manually check whether the answers match the original PDFs.